# TP 2 — Partie F & G : Tests de généralisation & Décision finale

Dans cette partie, le **jeu de test officiel (3 000 tweets) est enfin descellé**.
Nous appliquons les épreuves **F1 à F6** pour évaluer nos modèles en conditions réelles de production :
1. **F1. Décalage de prior (*Prior Shift*) :**
   - Comparaison du taux de sarcasme entre validation (17,29 %) et test (27,37 %).
   - Évaluation au seuil fixé sur `val` (sans tricher).
   - Tracé a posteriori de la **Courbe $F1 = f(\text{seuil})$ sur le test** (Figure obligatoire pour la **page 5 du compte-rendu**).
   - Sauvegarde des prédictions officielles dans `predictions/<modèle>_seed<k>.csv`.
2. **F2. Dialecte non vu (*Out-of-Distribution*) :**
   - Retrait des tweets égyptiens de `train`, réentraînement du modèle E et évaluation sur les tweets égyptiens du test (Question Q11).
3. **F3. Audit des raccourcis superficiels (*Shortcuts*) :**
   - Évaluation sur les sous-ensembles avec/sans URL, et réentraînement après suppression des URLs et mentions (Question Q2).
4. **F4. Indices du sarcasme (*Marker Ablation*) :**
   - Suppression des emojis, rires et ponctuations répétées sur les tweets sarcastiques bien prédits et mesure de la chute du rappel.
5. **F5. Significativité statistique :**
   - Intervalle de confiance à 95 % par Bootstrap (1 000 rééchantillonnages) et test de McNemar entre nos deux meilleurs modèles.
6. **F6. Analyse qualitative de 30 erreurs (15 FN / 15 FP) :**
   - Classification détaillée dans les 5 catégories imposées (Question Q12).
7. **Partie G. Tableau final et Matrice de confusion du modèle retenu.**

In [ ]:
# 1. Environnement et configuration
import os, sys
REPO = "/content/tp2-sarcasme-arabe"
if not os.path.exists(REPO):
    !git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!git pull

!pip install -q transformers arabert pyarabic emoji accelerate scipy matplotlib seaborn

In [ ]:
# 2. Chargement des données — Test enfin descellé !
import torch
import pandas as pd
import numpy as np
from src.seed import set_seed
from src.data import load_splits
from src.preprocess import dedupliquer_train
from src.results import ajouter_resultat, CHEMIN

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)

train_brut, val, test = load_splits(seed=42)
train = dedupliquer_train(train_brut, test, verbose=True)

print("\n--- PROPORTIONS DE SARCASME ---")
taux_val = val["sarcasm"].mean()
taux_test = test["sarcasm"].mean()
print(f"Taux de sarcasme sur VAL  : {taux_val:.2%}")
print(f"Taux de sarcasme sur TEST : {taux_test:.2%}  <-- DÉCALAGE DE PRIOR DE +10 POINTS !")

## 1. F1. Décalage de Prior & Courbe officielle $F1 = f(\text{seuil})$ sur le test
Nous évaluons notre meilleur modèle (MARBERTv2 multi-tâche) sur le test scellé au seuil fixé sur validation (`0.72`), puis nous traçons a posteriori la courbe complète.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from src.models_e import MultiTaskModel
from src.train_e import MultiTaskDataset, train_transformer, evaluate_transformer
from src.eval_f import plot_prior_shift_curve, save_official_predictions

MODEL_NAME = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Datasets
ds_tr = MultiTaskDataset(train["tweet"], train["sarcasm"].astype(int), train["sentiment"], tokenizer, max_len=64)
ds_va = MultiTaskDataset(val["tweet"], val["sarcasm"].astype(int), val["sentiment"], tokenizer, max_len=64)
ds_te = MultiTaskDataset(test["tweet"], test["sarcasm"].astype(int), test["sentiment"], tokenizer, max_len=64)

loader_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)
loader_va = DataLoader(ds_va, batch_size=32, shuffle=False)
loader_te = DataLoader(ds_te, batch_size=32, shuffle=False)

n_neg = (train["sarcasm"] == 0).sum()
n_pos = (train["sarcasm"] == 1).sum()
weights_sarc = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float32)

print("=== Entraînement du modèle final MARBERTv2 MT (lambda=0.3) ===")
set_seed(42)
best_model = MultiTaskModel(MODEL_NAME)
res_val = train_transformer(best_model, loader_tr, loader_va, device, weights_sarc, is_multitask=True, lam=0.3, epochs=4)

# Prédiction sur le TEST officiel
y_test_true, y_test_prob = evaluate_transformer(best_model, loader_te, device, is_multitask=True)
seuil_val_retenu = res_val["seuil"]

# F1 sur test avec le seuil choisi sur val + tracé de la courbe
f1_test_at_val_s, best_s_test, best_f1_test = plot_prior_shift_curve(
    y_test_true=y_test_true,
    y_test_prob=y_test_prob,
    seuil_val=seuil_val_retenu,
    model_name="MARBERTv2 Multi-tâche",
    save_path="results/prior_shift_f1_curve.png",
)

print(f"\nRésultat F1 sur le Test officiel avec seuil val ({seuil_val_retenu}) : {f1_test_at_val_s:.4f}")
print(f"Résultat F1 optimal a posteriori sur le Test (au seuil {best_s_test}) : {best_f1_test:.4f}")

# Sauvegarde des prédictions officielles au format exigé
y_test_pred = (y_test_prob >= seuil_val_retenu).astype(int)
save_official_predictions("MARBERTv2_multitask", 42, test, y_test_true, y_test_pred, y_test_prob)

## 2. F2. Généralisation sur dialecte non vu (Retrait de l'Égyptien de `train`)
Nous retirons tous les tweets égyptiens de `train`, ré-entraînons le modèle, et mesurons la chute de performance sur les tweets égyptiens du test.

In [ ]:
# Filtrage : train sans égyptien
train_sans_egypt = train[train["dialect"] != "egypt"].reset_index(drop=True)
test_egypt = test[test["dialect"] == "egypt"].reset_index(drop=True)

print(f"Train complet : {len(train)} | Train sans égyptien : {len(train_sans_egypt)} (-{len(train)-len(train_sans_egypt)} tweets retirés)")
print(f"Test égyptien seul : {len(test_egypt)} tweets")

# Dataloaders
ds_tr_no_egy = MultiTaskDataset(train_sans_egypt["tweet"], train_sans_egypt["sarcasm"].astype(int), train_sans_egypt["sentiment"], tokenizer, max_len=64)
ds_te_egy = MultiTaskDataset(test_egypt["tweet"], test_egypt["sarcasm"].astype(int), test_egypt["sentiment"], tokenizer, max_len=64)

loader_tr_no_egy = DataLoader(ds_tr_no_egy, batch_size=32, shuffle=True)
loader_te_egy = DataLoader(ds_te_egy, batch_size=32, shuffle=False)

set_seed(42)
model_no_egy = MultiTaskModel(MODEL_NAME)
_ = train_transformer(model_no_egy, loader_tr_no_egy, loader_va, device, weights_sarc, is_multitask=True, lam=0.3, epochs=4)

# Évaluation sur test egypt
y_egy_true, y_egy_prob_no = evaluate_transformer(model_no_egy, loader_te_egy, device, is_multitask=True)
f1_egypt_non_vu = f1_score(y_egy_true, (y_egy_prob_no >= seuil_val_retenu).astype(int), zero_division=0)

# Comparaison avec le modèle entraîné sur tout train
_, y_egy_prob_full = evaluate_transformer(best_model, loader_te_egy, device, is_multitask=True)
f1_egypt_vu = f1_score(y_egy_true, (y_egy_prob_full >= seuil_val_retenu).astype(int), zero_division=0)

print(f"\n--- F2. RÉSULTATS DIALECTE ÉGYPTIEN NON VU ---")
print(f"F1 sarc sur test egypt (modèle entraîné sur TOUT train) : {f1_egypt_vu:.4f}")
print(f"F1 sarc sur test egypt (modèle SANS égyptien en train)  : {f1_egypt_non_vu:.4f}")
print(f"Chute mesurée due à l'absence du dialecte : {f1_egypt_vu - f1_egypt_non_vu:.4f}")

del model_no_egy
torch.cuda.empty_cache()

## 3. F3. Audit des raccourcis superficiels (*Shortcuts*) — URLs & Mentions
Évaluation du modèle sur les sous-groupes avec et sans URL, et réentraînement après nettoyage des URLs et mentions.

In [ ]:
# Sous-groupes avec / sans URL sur le test officiel
has_url_test = test["tweet"].str.contains(r"http|https|t\.co", regex=True)

f1_avec_url = f1_score(y_test_true[has_url_test], y_test_pred[has_url_test], zero_division=0)
f1_sans_url = f1_score(y_test_true[~has_url_test], y_test_pred[~has_url_test], zero_division=0)

print("--- F3. SCORES DU MODÈLE SELON LA PRÉSENCE D'URL ---")
print(f"F1 sarc sur les tweets AVEC URL ({has_url_test.sum()} tweets) : {f1_avec_url:.4f}")
print(f"F1 sarc sur les tweets SANS URL ({(~has_url_test).sum()} tweets) : {f1_sans_url:.4f}")

## 4. F4. Dépendance aux indices de surface (*Marker Ablation*)
Sur les tweets sarcastiques du test **correctement détectés** (Vrais Positifs), nous supprimons artificiellement les emojis, les rires (`هههه`) et la ponctuation répétée, et nous mesurons la chute du rappel.

In [ ]:
from src.eval_f import ablate_sarcasm_markers

# Sélection des Vrais Positifs (sarcastiques correctement détectés)
vp_mask = (y_test_true == 1) & (y_test_pred == 1)
test_vp = test[vp_mask].copy()
print(f"Nombre de tweets sarcastiques correctement détectés au départ : {len(test_vp)}")

# Application de l'ablation des marqueurs de surface
test_vp["tweet_ablate"] = test_vp["tweet"].apply(ablate_sarcasm_markers)

# Ré-évaluation
ds_vp_ablate = MultiTaskDataset(test_vp["tweet_ablate"], [1]*len(test_vp), [0]*len(test_vp), tokenizer, max_len=64)
loader_vp_ablate = DataLoader(ds_vp_ablate, batch_size=32, shuffle=False)

_, probs_ablate = evaluate_transformer(best_model, loader_vp_ablate, device, is_multitask=True)
vp_apres_ablation = (probs_ablate >= seuil_val_retenu).sum()
taux_survie = vp_apres_ablation / len(test_vp)
chute_detection = 1.0 - taux_survie

print(f"Tweets toujours détectés après suppression des emojis, rires et ponctuations : {vp_apres_ablation} / {len(test_vp)}")
print(f"Taux de détection résiduel : {taux_survie:.2%}")
print(f"Chute du taux de détection due à l'ablation des marqueurs : {chute_detection:.2%}")

## 5. F5. Significativité Statistique : Bootstrap & Test de McNemar
Intervalle de confiance à 95 % par bootstrap (1 000 itérations) et test de McNemar comparant notre modèle multi-tâche E2 avec le modèle mono-tâche E1.

In [ ]:
from src.eval_f import bootstrap_f1_ci, mcnemar_test
from src.models_e import MonoTaskModel

# 1. Bootstrap 95% CI sur notre meilleur modèle E2
f1_mean_boot, ci_low, ci_high = bootstrap_f1_ci(y_test_true, y_test_prob, threshold=seuil_val_retenu, n_bootstraps=1000)
print(f"--- F5. BOOTSTRAP SUR MODÈLE E2 (1 000 rééchantillonnages) ---")
print(f"F1 sarc test : {f1_test_at_val_s:.4f} | Intervalle de Confiance à 95% : [{ci_low:.4f} , {ci_high:.4f}]")

# 2. Test de McNemar : Comparaison entre E2 Multi-tâche et E1 Mono-tâche AraBERT
print("\nChargement et prédiction du modèle E1 AraBERT mono-tâche sur test...")
tok_ara = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02-twitter")
ds_te_ara = MultiTaskDataset(test["tweet"], test["sarcasm"].astype(int), None, tok_ara, max_len=64)
loader_te_ara = DataLoader(ds_te_ara, batch_size=32, shuffle=False)

set_seed(42)
model_e1_comp = MonoTaskModel("aubmindlab/bert-base-arabertv02-twitter")
_ = train_transformer(model_e1_comp, loader_tr, loader_va, device, weights_sarc, is_multitask=False, epochs=4)
_, prob_e1_test = evaluate_transformer(model_e1_comp, loader_te_ara, device, is_multitask=False)
pred_e1_test = (prob_e1_test >= 0.50).astype(int)

b, c, stat, p_val = mcnemar_test(y_test_true, y_test_pred, pred_e1_test)
print(f"\n--- TEST DE MCNEMAR (E2 vs E1) ---")
print(f"E2 correct & E1 incorrect (b) : {b}")
print(f"E2 incorrect & E1 correct (c) : {c}")
print(f"Statistique chi2 : {stat:.4f} | p-value : {p_val:.4e}")
if p_val < 0.05:
    print("Conclusion : La différence de performance entre E2 et E1 est STATISTIQUEMENT SIGNIFICATIVE (p < 0.05) !")
else:
    print("Conclusion : La différence n'est pas statistiquement significative (p >= 0.05).")

del model_e1_comp
torch.cuda.empty_cache()

## 6. F6. Analyse Qualitative de 30 Erreurs (15 FN / 15 FP)
Extraction de 15 faux négatifs et 15 faux positifs pour classification dans les 5 catégories imposées par le sujet (Question Q12).

In [ ]:
# Identification des Faux Négatifs (Vrai sarc=1, Prédit 0) et Faux Positifs (Vrai sarc=0, Prédit 1)
test_eval = test.copy()
test_eval["y_true"] = y_test_true
test_eval["y_pred"] = y_test_pred
test_eval["prob"] = np.round(y_test_prob, 4)

fn_df = test_eval[(test_eval["y_true"] == 1) & (test_eval["y_pred"] == 0)].head(15)
fp_df = test_eval[(test_eval["y_true"] == 0) & (test_eval["y_pred"] == 1)].head(15)

fn_df["type_erreur"] = "Faux Négatif (Sarcasme manqué)"
fp_df["type_erreur"] = "Faux Positif (Fausse alerte)"

erreurs_30 = pd.concat([fn_df, fp_df]).reset_index(drop=True)
erreurs_30.to_csv("results/analyse_30_erreurs.csv", index=False)
print("30 erreurs exportées avec succès dans results/analyse_30_erreurs.csv !")
print(erreurs_30[["id", "type_erreur", "dialect", "sentiment", "prob", "tweet"]].to_string())

## 7. Partie G : Tableau comparatif final & Matrice de confusion
Génération de la matrice de confusion finale pour le compte-rendu.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test_true, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Prédit Normal (0)", "Prédit Sarcasme (1)"],
            yticklabels=["Vrai Normal (0)", "Vrai Sarcasme (1)"])
plt.title(f"Matrice de Confusion finale sur le Test officiel\nMARBERTv2 MT (seuil = {seuil_val_retenu})", fontsize=12, fontweight="bold")
plt.ylabel("Vérité terrain")
plt.xlabel("Prédiction")
plt.tight_layout()
plt.savefig("results/matrice_confusion_test.png", dpi=300)
print("Matrice de confusion sauvegardée dans results/matrice_confusion_test.png !")
plt.show()